In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import pandas_ta as ta
import sklearn
import torch

print("pandas:    ", pd.__version__)
print("numpy:     ", np.__version__)
print("yfinance:  ", yf.__version__)
print("pandas_ta: ", ta.version)
print("sklearn:   ", sklearn.__version__)
print("torch:     ", torch.__version__)
print("\nAll systems go")

pandas:     3.0.2
numpy:      2.2.6
yfinance:   1.3.0
pandas_ta:  0.4.71b0
sklearn:    1.8.0
torch:      2.11.0+cpu

All systems go


In [2]:
import yfinance as yf
import pandas as pd

# Fetch 5 years of daily AAPL data
aapl = yf.download("AAPL", period="5y", interval="1d", auto_adjust=False)

print("Shape:", aapl.shape)
print("\nFirst 5 rows:")
aapl.head()

[*********************100%***********************]  1 of 1 completed

Shape: (1255, 6)

First 5 rows:


Price,Adj Close,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,,
2021-04-21,130.028397,133.500000,133.750000,131.300003,132.360001,68847100
2021-04-22,128.508972,131.940002,134.149994,131.410004,133.039993,84566500
2021-04-23,130.827042,134.320007,135.119995,132.160004,132.160004,78657500
2021-04-26,131.216690,134.720001,135.059998,133.559998,134.830002,66905100
2021-04-27,130.895264,134.389999,135.410004,134.110001,135.009995,66015800


In [3]:
# Re-fetch AAPL with auto-adjust enabled for simplicity
# Now "Close" = adjusted close (handles splits/dividends automatically)
aapl = yf.download(
    "AAPL",
    period="5y",
    interval="1d",
    auto_adjust=True  # key flag: Close is now split/dividend-adjusted
)

# Handle the MultiIndex columns that yfinance now returns
if isinstance(aapl.columns, pd.MultiIndex):
    aapl.columns = aapl.columns.get_level_values(0)

print("Shape:", aapl.shape)
print("\nDate range:", aapl.index.min().date(), "→", aapl.index.max().date())
print("\nColumns:", list(aapl.columns))
print("\nFirst 3 rows:")
print(aapl.head(3))
print("\nLast 3 rows:")
print(aapl.tail(3))
print("\nMissing values per column:")
print(aapl.isna().sum())

[*********************100%***********************]  1 of 1 completed

Shape: (1255, 5)

Date range: 2021-04-21 → 2026-04-20

Columns: ['Close', 'High', 'Low', 'Open', 'Volume']

First 3 rows:
Price            Close        High         Low        Open    Volume
Date                                                                
2021-04-21  130.028427  130.271926  127.885640  128.918073  68847100
2021-04-22  128.508972  130.661494  127.992756  129.580358  84566500
2021-04-23  130.827118  131.606303  128.723284  128.723284  78657500

Last 3 rows:
Price            Close        High         Low        Open    Volume
Date                                                                
2026-04-16  263.399994  267.160004  261.269989  266.799988  43323100
2026-04-17  270.230011  272.299988  266.720001  266.959991  61314800
2026-04-20  273.049988  274.274994  270.290009  270.329987  34667241

Missing values per column:
Price
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64


In [6]:
import pandas_ta as ta

# Work on a copy so original raw data is preserved
df = aapl.copy()

# --- Feature 1: Daily Return (% change from previous day's close) ---
df["daily_return"] = df["Close"].pct_change()

# --- Feature 2: Simple Moving Average (10-day and 20-day) ---
df["sma_10"] = df["Close"].rolling(window=10).mean()
df["sma_20"] = df["Close"].rolling(window=20).mean()

# --- Feature 3: Volatility (rolling std of daily returns, 10-day window) ---
df["volatility_10"] = df["daily_return"].rolling(window=10).std()

# --- Feature 4: Momentum (% change over 10 days) ---
df["momentum_10"] = df["Close"].pct_change(periods=10)

# --- Feature 5: RSI (Relative Strength Index, 14-day — industry standard) ---
df["rsi_14"] = ta.rsi(df["Close"], length=14)

# Inspect
print("Shape:", df.shape)
print("\nColumns now:", list(df.columns))
print("\nLast 5 rows (most recent data):")
print(df.tail())
print("\nMissing values per column:")
print(df.isna().sum())

Shape: (1255, 11)

Columns now: ['Close', 'High', 'Low', 'Open', 'Volume', 'daily_return', 'sma_10', 'sma_20', 'volatility_10', 'momentum_10', 'rsi_14']

Last 5 rows (most recent data):
Price            Close        High         Low        Open    Volume  \
Date                                                                   
2026-04-14  258.829987  261.929993  257.190002  259.250000  48370700   
2026-04-15  266.429993  266.559998  257.809998  258.160004  49913500   
2026-04-16  263.399994  267.160004  261.269989  266.799988  43323100   
2026-04-17  270.230011  272.299988  266.720001  266.959991  61314800   
2026-04-20  273.049988  274.274994  270.290009  270.329987  34667241   

Price       daily_return      sma_10      sma_20  volatility_10  momentum_10  \
Date                                                                           
2026-04-14     -0.001428  257.559998  254.039500       0.013874     0.049467   
2026-04-15      0.029363  258.823997  254.649500       0.013938     0

In [7]:
# --- Create targets using shift(-1) to look one day into the future ---

# Regression target: tomorrow's daily return (as a decimal)
# On row=2026-04-16, this gives the return that will be realized on 2026-04-17
df["target_return"] = df["daily_return"].shift(-1)

# Classification target: 1 if tomorrow's return is positive (UP), 0 otherwise (DOWN)
# Derived directly from target_return to guarantee consistency between the two targets
df["target_direction"] = (df["target_return"] > 0).astype(int)

# Inspect — focus on the LAST few rows to verify the target-shift alignment
print("Last 5 rows (pay attention to how target_return aligns with tomorrow's daily_return):")
print(df[["Close", "daily_return", "target_return", "target_direction"]].tail())

print("\nMissing values in targets:")
print(df[["target_return", "target_direction"]].isna().sum())

Last 5 rows (pay attention to how target_return aligns with tomorrow's daily_return):
Price            Close  daily_return  target_return  target_direction
Date                                                                 
2026-04-14  258.829987     -0.001428       0.029363                 1
2026-04-15  266.429993      0.029363      -0.011373                 0
2026-04-16  263.399994     -0.011373       0.025930                 1
2026-04-17  270.230011      0.025930       0.010435                 1
2026-04-20  273.049988      0.010435            NaN                 0

Missing values in targets:
Price
target_return       1
target_direction    0
dtype: int64


In [8]:
# Before cleaning — capture the "before" state for a clean diff
print("Before cleanup:")
print(f"  Rows: {len(df)}")
print(f"  Total NaNs: {df.isna().sum().sum()}")

# Drop all rows that contain ANY NaN across feature or target columns
df_clean = df.dropna().copy()

print("\nAfter cleanup:")
print(f"  Rows: {len(df_clean)}")
print(f"  Total NaNs: {df_clean.isna().sum().sum()}")
print(f"  Rows dropped: {len(df) - len(df_clean)}")

print("\nDate range of cleaned data:")
print(f"  {df_clean.index.min().date()} → {df_clean.index.max().date()}")

print("\nFirst 3 rows of cleaned data (should start with no NaNs):")
print(df_clean.head(3))

print("\nClass balance (UP=1 vs DOWN=0):")
print(df_clean["target_direction"].value_counts())
print(f"\nUP rate: {df_clean['target_direction'].mean():.3f}")

Before cleanup:
  Rows: 1255
  Total NaNs: 51

After cleanup:
  Rows: 1235
  Total NaNs: 0
  Rows dropped: 20

Date range of cleaned data:
  2021-05-18 → 2026-04-17

First 3 rows of cleaned data (should start with no NaNs):
Price            Close        High         Low        Open    Volume  \
Date                                                                   
2021-05-18  121.809921  123.897812  121.741626  123.478282  63342900   
2021-05-19  121.653786  121.878181  119.868345  120.161043  92612000   
2021-05-20  124.209991  124.610012  122.053806  122.180645  76857100   

Price       daily_return      sma_10      sma_20  volatility_10  momentum_10  \
Date                                                                           
2021-05-18     -0.011246  123.583903  126.454539       0.016407    -0.021806   
2021-05-19     -0.001282  123.272399  126.035807       0.016350    -0.024966   
2021-05-20      0.021012  123.056779  125.820858       0.017373    -0.017063   

Price         

In [9]:
def fetch_ticker_data(ticker: str, period: str = "5y") -> pd.DataFrame:
    """
    Download OHLCV data for a single ticker.
    Returns a DataFrame indexed by Date with columns: Close, High, Low, Open, Volume.
    Close is already split/dividend-adjusted (auto_adjust=True).
    """
    df = yf.download(ticker, period=period, interval="1d", auto_adjust=True, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add technical indicators as new columns:
    daily_return, sma_10, sma_20, volatility_10, momentum_10, rsi_14.
    """
    df = df.copy()
    df["daily_return"] = df["Close"].pct_change()
    df["sma_10"] = df["Close"].rolling(window=10).mean()
    df["sma_20"] = df["Close"].rolling(window=20).mean()
    df["volatility_10"] = df["daily_return"].rolling(window=10).std()
    df["momentum_10"] = df["Close"].pct_change(periods=10)
    df["rsi_14"] = ta.rsi(df["Close"], length=14)
    return df


def add_targets(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add prediction targets as new columns:
      target_return     — tomorrow's daily return (regression target)
      target_direction  — 1 if tomorrow is UP, 0 if DOWN (classification target)
    Uses shift(-1) to look exactly one day into the future.
    """
    df = df.copy()
    df["target_return"] = df["daily_return"].shift(-1)
    df["target_direction"] = (df["target_return"] > 0).astype(int)
    return df


def process_ticker(ticker: str, period: str = "5y") -> pd.DataFrame:
    """
    Full single-ticker pipeline: fetch → features → targets → drop NaNs → tag.
    Returns a clean, modeling-ready DataFrame with a 'ticker' column added.
    """
    df = fetch_ticker_data(ticker, period=period)
    df = add_features(df)
    df = add_targets(df)
    df = df.dropna().copy()
    df["ticker"] = ticker
    return df


# Sanity check: re-run the AAPL pipeline through the new functions and confirm
# the output matches what we already validated cell-by-cell above.
aapl_refactored = process_ticker("AAPL", period="5y")
print(f"AAPL refactored pipeline: {len(aapl_refactored)} rows, {len(aapl_refactored.columns)} columns")
print(f"Columns: {list(aapl_refactored.columns)}")
print(f"Date range: {aapl_refactored.index.min().date()} → {aapl_refactored.index.max().date()}")
print(f"\nLast 3 rows:")
print(aapl_refactored.tail(3))

AAPL refactored pipeline: 1235 rows, 14 columns
Columns: ['Close', 'High', 'Low', 'Open', 'Volume', 'daily_return', 'sma_10', 'sma_20', 'volatility_10', 'momentum_10', 'rsi_14', 'target_return', 'target_direction', 'ticker']
Date range: 2021-05-18 → 2026-04-17

Last 3 rows:
Price            Close        High         Low        Open    Volume  \
Date                                                                   
2026-04-15  266.429993  266.559998  257.809998  258.160004  49913500   
2026-04-16  263.399994  267.160004  261.269989  266.799988  43323100   
2026-04-17  270.230011  272.299988  266.720001  266.959991  61436200   

Price       daily_return      sma_10      sma_20  volatility_10  momentum_10  \
Date                                                                           
2026-04-15      0.029363  258.823997  254.649500       0.013938     0.049805   
2026-04-16     -0.011373  259.600996  255.322499       0.014815     0.030395   
2026-04-17      0.025930  261.031998  256.38

In [10]:
# Define the ticker universe — diversified across sectors per your spec
TICKERS = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "TSLA",   # Tech
    "JPM", "GS", "BAC",                         # Finance
    "JNJ", "XOM",                               # Healthcare + Energy
]

# Process each ticker independently
all_ticker_dfs = []
for ticker in TICKERS:
    try:
        df_t = process_ticker(ticker, period="5y")
        print(f"{ticker:6s} — {len(df_t):5d} rows, "
              f"UP rate: {df_t['target_direction'].mean():.3f}")
        all_ticker_dfs.append(df_t)
    except Exception as e:
        print(f"{ticker:6s} — FAILED: {e}")

# Combine into one big DataFrame
combined_df = pd.concat(all_ticker_dfs, ignore_index=False)

print(f"\n")
print(f"Combined dataset: {len(combined_df)} rows across {combined_df['ticker'].nunique()} tickers")
print(f"Overall UP rate: {combined_df['target_direction'].mean():.3f}")
print(f"Per-ticker row counts:")
print(combined_df["ticker"].value_counts())

AAPL   —  1235 rows, UP rate: 0.531
MSFT   —  1235 rows, UP rate: 0.519
GOOGL  —  1235 rows, UP rate: 0.534
AMZN   —  1235 rows, UP rate: 0.514
TSLA   —  1235 rows, UP rate: 0.518
JPM    —  1235 rows, UP rate: 0.534
GS     —  1235 rows, UP rate: 0.527
BAC    —  1235 rows, UP rate: 0.509
JNJ    —  1235 rows, UP rate: 0.514
XOM    —  1235 rows, UP rate: 0.535


Combined dataset: 12350 rows across 10 tickers
Overall UP rate: 0.524
Per-ticker row counts:
ticker
AAPL     1235
MSFT     1235
GOOGL    1235
AMZN     1235
TSLA     1235
JPM      1235
GS       1235
BAC      1235
JNJ      1235
XOM      1235
Name: count, dtype: int64


In [12]:
import os
from pathlib import Path

# Define output path — relative to the project root
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / "nexttick_dataset.csv"

# Save — keep the Date index so chronological order is preserved
combined_df.to_csv(output_file, index=True)

# Verify the save worked
file_size_mb = os.path.getsize(output_file) / (1024 * 1024)
print(f"   Saved: {output_file}")
print(f"   File size: {file_size_mb:.2f} MB")
print(f"   Rows: {len(combined_df):,}")
print(f"   Columns: {len(combined_df.columns)}")

# Sanity check: reload and verify
reloaded = pd.read_csv(output_file, index_col="Date", parse_dates=True)
print(f"\n   Reload test: {len(reloaded)} rows, matches original: {len(reloaded) == len(combined_df)}")
print(f"\nFirst 2 rows of reloaded data:")
print(reloaded.head(2))

   Saved: ..\data\processed\nexttick_dataset.csv
   File size: 2.82 MB
   Rows: 12,350
   Columns: 14

   Reload test: 12350 rows, matches original: True

First 2 rows of reloaded data:
                 Close        High         Low        Open    Volume  \
Date                                                                   
2021-05-18  121.809883  123.897773  121.741588  123.478243  63342900   
2021-05-19  121.653801  121.878196  119.868360  120.161058  92612000   

            daily_return      sma_10      sma_20  volatility_10  momentum_10  \
Date                                                                           
2021-05-18     -0.011246  123.583904  126.454544       0.016407    -0.021806   
2021-05-19     -0.001281  123.272401  126.035814       0.016350    -0.024966   

               rsi_14  target_return  target_direction ticker  
Date                                                           
2021-05-18  28.148615      -0.001281                 0   AAPL  
2021-05-19  